# Parameter Tuning

Interactive parameter tuning with real-time feedback on prediction accuracy.

Current rating snapshots are not historical pregame evidence. The scoring functions leave qualification unavailable for past or started games. Do not interpret these comparisons as validation of historical playoff stakes, or use them to tune those stakes. Strength and championship inputs also lack a historical evidence guarantee. The evaluation time is explicit, and team lookups include the game season.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output

from hype_score.database import get_engine, load_games, load_team_ratings_for_games, load_pregame_odds, load_odds_snapshots
from hype_score.config import HypeConfig, SportWeights
from hype_score.scoring import calculate_hype_scores_batch
from hype_score.excitement import calculate_excitement_batch
from hype_score.validation import compare_hype_vs_excitement

## Load Data

In [ ]:
evaluated_at = pd.Timestamp.now(tz="UTC")
engine = get_engine()

games = load_games(engine, status='final', limit=1000)
pregame_odds = load_pregame_odds(engine, game_ids=games['id'].tolist())
ratings = load_team_ratings_for_games(engine, games, evaluated_at=evaluated_at)
all_snapshots = load_odds_snapshots(engine, game_ids=games['id'].tolist())

excitement_results = calculate_excitement_batch(all_snapshots)

print(f"Ready with {len(games):,} games, {len(excitement_results):,} with excitement data")

## Baseline Metrics

In [ ]:
baseline_config = HypeConfig.default()
baseline_hype = calculate_hype_scores_batch(games, pregame_odds, ratings, baseline_config, evaluated_at=evaluated_at)
baseline_comparison = compare_hype_vs_excitement(baseline_hype, excitement_results)

print("Baseline Metrics:")
print(f"  Correlation: {baseline_comparison['correlation']:.3f}")
print(f"  MAE: {baseline_comparison['mae']:.2f}")
print(f"  Classification Accuracy: {baseline_comparison.get('classification_accuracy', 0)*100:.1f}%")

## Interactive Sport-Specific Tuning

In [ ]:
sport_sliders = {}
default_config = HypeConfig.default()

for sport in ['nfl', 'nba', 'mlb', 'nhl']:
    default_weights = default_config.get_sport_weights(sport)
    sport_sliders[sport] = {
        'stakes': widgets.FloatSlider(value=default_weights.stakes_weight, min=0, max=1, step=0.05, description=f'{sport.upper()} Stakes:'),
        'quality': widgets.FloatSlider(value=default_weights.quality_weight, min=0, max=1, step=0.05, description=f'{sport.upper()} Quality:'),
        'closeness': widgets.FloatSlider(value=default_weights.closeness_weight, min=0, max=1, step=0.05, description=f'{sport.upper()} Close:'),
    }

output = widgets.Output()

def recalculate(change=None):
    with output:
        clear_output(wait=True)
        
        custom_config = HypeConfig.default()
        
        for sport, sliders in sport_sliders.items():
            total = sliders['stakes'].value + sliders['quality'].value + sliders['closeness'].value
            if total == 0:
                continue
            custom_config.sport_weights[sport] = SportWeights(
                sliders['stakes'].value / total,
                sliders['quality'].value / total,
                sliders['closeness'].value / total
            )
        
        new_hype = calculate_hype_scores_batch(games, pregame_odds, ratings, custom_config, evaluated_at=evaluated_at)
        new_comparison = compare_hype_vs_excitement(new_hype, excitement_results)
        
        print("Current vs Baseline:")
        print(f"  Correlation: {new_comparison['correlation']:.3f} (baseline: {baseline_comparison['correlation']:.3f})")
        print(f"  MAE: {new_comparison['mae']:.2f} (baseline: {baseline_comparison['mae']:.2f})")
        print(f"  Classification Accuracy: {new_comparison.get('classification_accuracy', 0)*100:.1f}% (baseline: {baseline_comparison.get('classification_accuracy', 0)*100:.1f}%)")
        
        corr_diff = new_comparison['correlation'] - baseline_comparison['correlation']
        mae_diff = baseline_comparison['mae'] - new_comparison['mae']   
        
        if corr_diff > 0.01:
            print(f"  ↑ Correlation improved by {corr_diff:.3f}")
        elif corr_diff < -0.01:
            print(f"  ↓ Correlation decreased by {abs(corr_diff):.3f}")
            
        if mae_diff > 0.5:
            print(f"  ↑ MAE improved by {mae_diff:.2f}")
        elif mae_diff < -0.5:
            print(f"  ↓ MAE worsened by {abs(mae_diff):.2f}")

for sport, sliders in sport_sliders.items():
    for slider in sliders.values():
        slider.observe(recalculate, names='value')

sport_tabs = widgets.Tab()
tab_children = []
for sport, sliders in sport_sliders.items():
    tab_children.append(widgets.VBox([sliders['stakes'], sliders['quality'], sliders['closeness']]))
sport_tabs.children = tab_children
for i, sport in enumerate(sport_sliders.keys()):
    sport_tabs.set_title(i, sport.upper())

display(sport_tabs)
display(output)
recalculate()

## Grid Search Optimization

In [ ]:
from itertools import product
from tqdm.auto import tqdm

def grid_search_sport(sport: str, games_df, pregame_odds_df, ratings_df, excitement_df):
    sport_games = games_df[games_df['sport'] == sport]
    if len(sport_games) == 0:
        return None
    
    weight_options = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6]
    
    best_correlation = -1
    best_weights = None
    results = []
    
    for stakes, quality, closeness in product(weight_options, repeat=3):
        total = stakes + quality + closeness
        if total == 0:
            continue
        
        s, q, c = stakes/total, quality/total, closeness/total
        
        config = HypeConfig.default()
        config.sport_weights[sport] = SportWeights(s, q, c)
        
        hype = calculate_hype_scores_batch(sport_games, pregame_odds_df, ratings_df, config, evaluated_at=evaluated_at)
        comparison = compare_hype_vs_excitement(hype, excitement_df)
        
        if comparison['correlation'] is not None:
            results.append({
                'stakes': s,
                'quality': q,
                'closeness': c,
                'correlation': comparison['correlation'],
                'mae': comparison['mae']
            })
            
            if comparison['correlation'] > best_correlation:
                best_correlation = comparison['correlation']
                best_weights = (s, q, c)
    
    return {
        'sport': sport,
        'best_weights': best_weights,
        'best_correlation': best_correlation,
        'all_results': pd.DataFrame(results)
    }

In [ ]:
print("Running grid search for optimal weights per sport...")

sports_to_search = games['sport'].unique()
optimal_weights = {}

for sport in tqdm(sports_to_search):
    result = grid_search_sport(sport, games, pregame_odds, ratings, excitement_results)
    if result:
        optimal_weights[sport] = result
        print(f"{sport.upper()}: stakes={result['best_weights'][0]:.2f}, quality={result['best_weights'][1]:.2f}, closeness={result['best_weights'][2]:.2f} (corr={result['best_correlation']:.3f})")

In [ ]:
print("\nOptimal vs Default Weights Comparison:")
print(f"{'Sport':<8} {'Default':<30} {'Optimal':<30} {'Δ Corr':>8}")
print("-" * 80)

for sport, result in optimal_weights.items():
    default = baseline_config.get_sport_weights(sport)
    optimal = result['best_weights']
    
    default_str = f"s={default.stakes_weight:.2f} q={default.quality_weight:.2f} c={default.closeness_weight:.2f}"
    optimal_str = f"s={optimal[0]:.2f} q={optimal[1]:.2f} c={optimal[2]:.2f}"
    
    sport_baseline = baseline_comparison['merged_data'][baseline_comparison['merged_data']['sport'] == sport]
    if len(sport_baseline) >= 2:
        from scipy import stats
        baseline_corr, _ = stats.pearsonr(sport_baseline['hype_score'], sport_baseline['volatility_score'])
        delta = result['best_correlation'] - baseline_corr
        print(f"{sport.upper():<8} {default_str:<30} {optimal_str:<30} {delta:>+8.3f}")

## Export Optimized Config

In [ ]:
toml_content = """# Optimized hype score configuration
# Generated from grid search optimization

"""

for sport, result in optimal_weights.items():
    s, q, c = result['best_weights']
    toml_content += f"""[sport_weights.{sport}]
stakes = {s:.2f}
quality = {q:.2f}
closeness = {c:.2f}

"""

toml_content += """[normalization]
elo_min = 1300
elo_max = 1700
bpi_min = -20
bpi_max = 20
nhl_points_min = 60
nhl_points_max = 130

[stakes]
bubble_factor_max = 60
champ_bonus_weight = 0.2
season_progress_max = 20
"""

print(toml_content)

